# SSL-MAE / I-JEPA Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimchev/SSL-MAE/blob/master/notebooks/colab_demo.ipynb)

Semi-supervised learning with masked autoencoders for remote sensing image classification.

## 1. Setup

In [ ]:
%cd /content
!rm -rf SSL-MAE
!git clone https://github.com/marjanstoimchev/SSL-MAE.git
%cd SSL-MAE
!pip install -q -r requirements.txt

In [ ]:
import os, sys, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightning as L

sys.path.insert(0, '.')

from src.utils.config import load_config
from src.utils.evaluate import evaluate, collect_targets
from src.utils.metrics import calculate_mlc_metrics, calculate_mcc_metrics
from src.models import ssl_mae, ssl_ijepa
from src.data.datamodule import SSLMAEDataModule
from src.trainers import MAELearner, IJEPALearner
from src.trainers.callbacks import EarlyStopping_, RichProgressBar_
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'figure.dpi': 100,
    'axes.titleweight': 'bold',
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Configuration

In [ ]:
MODEL_TYPE = "mae"                # "mae" or "ijepa"
DATASET = "ucm_mlc"               # dataset name
FRACTIONS = [0.01, 0.05, 0.1, 0.25]  # fractions to show results for
FRACTION_LABELED = 0.1            # fraction for training / predictions demo
EPOCHS = 20                       # epochs (only used if SKIP_TRAINING=False)
BATCH_SIZE = 16

SKIP_TRAINING = True              # True = show pre-computed results, False = train

MODES = ["semi_supervised", "supervised", "supervised_baseline"]

METRICS = [
    {"key": "average auprc",   "title": "Average AUPRC"},
    {"key": "ranking loss",    "title": "Ranking Loss"},
    {"key": "micro f1",        "title": "Micro F1"},
    {"key": "subset accuracy", "title": "Subset Accuracy"},
]

MODE_COLORS = {
    "semi_supervised": "#2176AE",
    "supervised": "#E8871E",
    "supervised_baseline": "#57A773",
}
MODE_LABELS = {
    "semi_supervised": "Semi-Supervised",
    "supervised": "Supervised",
    "supervised_baseline": "Baseline",
}

EXP_NAME = f"{MODEL_TYPE}_{DATASET}"
SAVED_DIR = "saved_models"

## 3. Load or train

In [ ]:
SAVED_DIR = "saved_models"
results = {}

# Results are committed in the repo (metrics.txt only, no model weights)
# Load them directly — no download needed for visualization
for mode in MODES:
    mode_results = {}
    for frac in FRACTIONS:
        p = os.path.join(SAVED_DIR, EXP_NAME, mode, f"fl_{frac}", "results", "metrics.txt")
        if os.path.exists(p):
            mode_results[frac] = pd.read_csv(p, sep="\t", index_col=0)
    if mode_results:
        results[mode] = mode_results
        fracs_str = ", ".join(f"{int(f*100)}%" for f in mode_results)
        print(f"  {MODE_LABELS[mode]:25s} {fracs_str}")

if results:
    print(f"\nLoaded results for {len(results)} modes.")
else:
    print("No pre-computed results found. Set SKIP_TRAINING = False below to train.")

## 4. Results comparison

In [ ]:
# Summary table
rows = []
for mode, frac_results in results.items():
    for frac, df in sorted(frac_results.items()):
        row = {"Method": MODE_LABELS[mode], "Fraction": f"{int(frac*100)}%"}
        for m in METRICS:
            if m["key"] in df.index:
                row[m["title"]] = f"{df.loc[m['key'], 'Metric Value']:.4f}"
        rows.append(row)

if rows:
    summary = pd.DataFrame(rows).set_index(["Method", "Fraction"])
    display(summary)
else:
    print("No results.")

## 5. Learning curves + bar plots

In [ ]:
active_modes = [m for m in MODES if m in results]
active_fracs = sorted(set(f for mr in results.values() for f in mr.keys()))

# --- Line plots (learning curves) ---
n_metrics = len(METRICS)
n_cols = min(2, n_metrics)
n_rows = int(np.ceil(n_metrics / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = np.atleast_2d(axes).flatten()

x_pos = np.arange(len(active_fracs))
x_labels = [f"{int(f*100)}%" for f in active_fracs]

for ax_idx, m in enumerate(METRICS):
    ax = axes[ax_idx]
    for mode in active_modes:
        vals = [results[mode].get(f, pd.DataFrame()).loc[m["key"], "Metric Value"]
                if f in results[mode] and m["key"] in results[mode][f].index else np.nan
                for f in active_fracs]
        vals = np.array(vals)
        valid = ~np.isnan(vals)
        if valid.any():
            ax.plot(x_pos[valid], vals[valid], marker='o', markersize=5,
                    color=MODE_COLORS[mode], label=MODE_LABELS[mode],
                    markeredgecolor='white', markeredgewidth=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.set_xlabel("Fraction of labeled data")
    ax.set_title(m["title"])
    if ax_idx == 0:
        ax.legend(frameon=True, edgecolor='none', facecolor='white')

for idx in range(n_metrics, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(f"{MODEL_TYPE.upper()} — Learning Curves", y=1.02)
plt.tight_layout()
plt.show()

# --- Bar plots per fraction ---
for m in METRICS:
    n_fracs = len(active_fracs)
    fig, axes = plt.subplots(1, n_fracs, figsize=(2.2 * n_fracs + 2.5, 3.2), sharey=False)
    if n_fracs == 1: axes = [axes]

    colors = [MODE_COLORS[mode] for mode in active_modes]

    for i, frac in enumerate(active_fracs):
        ax = axes[i]
        vals = []
        bar_colors = []
        for mode in active_modes:
            if frac in results.get(mode, {}) and m["key"] in results[mode][frac].index:
                vals.append(results[mode][frac].loc[m["key"], "Metric Value"])
                bar_colors.append(MODE_COLORS[mode])
            else:
                vals.append(np.nan)
                bar_colors.append('#ccc')

        valid = [not np.isnan(v) for v in vals]
        x = np.arange(len(active_modes))
        ax.bar([x[j] for j in range(len(active_modes)) if valid[j]],
               [vals[j] for j in range(len(active_modes)) if valid[j]],
               color=[bar_colors[j] for j in range(len(active_modes)) if valid[j]],
               edgecolor='white', linewidth=0.5, width=0.7)

        valid_vals = [v for v in vals if not np.isnan(v)]
        if valid_vals:
            vmin, vmax = min(valid_vals), max(valid_vals)
            pad = max((vmax - vmin) * 0.2, 0.02)
            ax.set_ylim(max(0, vmin - pad), vmax + pad)

        ax.set_xticks([])
        ax.set_title(f"{int(frac*100)}%", fontsize=10)
        if i == 0:
            ax.set_ylabel(m["title"], fontsize=10)

    axes[-1].legend(
        [plt.Rectangle((0,0),1,1, fc=MODE_COLORS[mode]) for mode in active_modes],
        [MODE_LABELS[mode] for mode in active_modes],
        frameon=True, edgecolor='none', facecolor='white',
        fontsize=8, loc='center left', bbox_to_anchor=(1.05, 0.5),
    )
    fig.suptitle(m["title"], fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Train (optional)\n\nSet `SKIP_TRAINING = False` to train a specific fraction. Otherwise skip to predictions.

In [ ]:
if not SKIP_TRAINING:
    TRAIN_MODE = "semi_supervised"  # mode to train
    TRAIN_FRAC = FRACTION_LABELED   # fraction to train

    cfg = load_config(f"configs/{MODEL_TYPE}/{DATASET}.yaml", cli_args=[
        f"training.mode={TRAIN_MODE}", f"training.epochs={EPOCHS}",
        f"data.batch_size={BATCH_SIZE}", f"data.fraction_labeled={TRAIN_FRAC}",
        "data.num_workers=2", "training.devices=[0]",
    ])
    L.seed_everything(cfg.experiment.seed, workers=True)

    if MODEL_TYPE == "ijepa":
        model = ssl_ijepa(
            architecture=cfg.model.architecture, model_size=cfg.model.model_size,
            learning_task=cfg.data.learning_task, n_classes=cfg.data.n_classes, w=cfg.model.w,
            predictor_embed_dim=cfg.model.get('predictor_embed_dim', 384),
            predictor_depth=cfg.model.get('predictor_depth', 6),
            ema_decay=cfg.model.get('ema_target_decay', 0.996),
        )
        learner = IJEPALearner(model, cfg)
    else:
        model = ssl_mae(
            architecture=cfg.model.architecture, model_size=cfg.model.model_size,
            learning_task=cfg.data.learning_task, n_classes=cfg.data.n_classes, w=cfg.model.w,
        )
        learner = MAELearner(model, cfg)

    dm = SSLMAEDataModule(cfg)
    trainer = L.Trainer(
        max_epochs=EPOCHS, accelerator='auto', devices=[0],
        precision=cfg.training.precision, enable_checkpointing=False, logger=False,
        callbacks=[EarlyStopping_(metric='val_loss', mode='min', patience=cfg.training.patience),
                   RichProgressBar_()],
    )
    trainer.fit(learner, datamodule=dm)

    save_dir = os.path.join(SAVED_DIR, EXP_NAME, TRAIN_MODE, f"fl_{TRAIN_FRAC}")
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(save_dir, "final_model.pt"))
    print(f"Model saved to {save_dir}/final_model.pt")

    # Evaluate
    dm.setup()
    evaluate(trainer, learner, dm, cfg,
             results_dir=os.path.join(save_dir, "results"))
else:
    print("Skipping training (SKIP_TRAINING = True). Change to False to train.")

## 7. Load models for predictions

In [ ]:
PRED_FRAC = 0.1

# Google Drive file IDs for model weights (per mode)
GDRIVE_MODEL_IDS = {
    "semi_supervised":     "1qdc65_sTwuMfVR0mf-j_1Ww7kXNiI3Fz",
    "supervised":          "1JHhlaLY2BHXc46S005uNCqq1WwJglPm8",
    "supervised_baseline": "1v9n9FmpxuT_qczWT7mOQBAUbCbD-agoQ",
}

# UCM dataset class names
CLASS_NAMES = ['airplane', 'bare soil', 'buildings', 'cars', 'chaparral',
               'court', 'dock', 'field', 'grass', 'mobile home', 'pavement',
               'sand', 'sea', 'ship', 'tanks', 'trees', 'water']

def unnormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Download and load all 3 models
all_probas = {}
dm = None

for mode in MODES:
    model_dir = os.path.join(SAVED_DIR, EXP_NAME, mode, f"fl_{PRED_FRAC}")
    model_path = os.path.join(model_dir, "final_model.pt")

    # Download if not exists
    if not os.path.exists(model_path):
        file_id = GDRIVE_MODEL_IDS.get(mode)
        if file_id:
            !pip install -q gdown
            import gdown
            os.makedirs(model_dir, exist_ok=True)
            print(f"Downloading {MODE_LABELS[mode]}...")
            gdown.download(id=file_id, output=model_path, quiet=False)
        else:
            print(f"No Drive ID for {mode}, skipping")
            continue

    # Load model
    cfg = load_config(f"configs/{MODEL_TYPE}/{DATASET}.yaml", cli_args=[
        f"training.mode={mode}", f"data.fraction_labeled={PRED_FRAC}",
        f"data.batch_size={BATCH_SIZE}", "data.num_workers=2", "training.devices=[0]",
    ])

    if MODEL_TYPE == "ijepa":
        model = ssl_ijepa(
            architecture=cfg.model.architecture, model_size=cfg.model.model_size,
            learning_task=cfg.data.learning_task, n_classes=cfg.data.n_classes, w=cfg.model.w,
            predictor_embed_dim=cfg.model.get('predictor_embed_dim', 384),
            predictor_depth=cfg.model.get('predictor_depth', 6),
            ema_decay=cfg.model.get('ema_target_decay', 0.996),
        )
        learner_cls = IJEPALearner
    else:
        model = ssl_mae(
            architecture=cfg.model.architecture, model_size=cfg.model.model_size,
            learning_task=cfg.data.learning_task, n_classes=cfg.data.n_classes, w=cfg.model.w,
        )
        learner_cls = MAELearner

    state = torch.load(model_path, map_location='cpu')
    model.load_state_dict(state)
    model.eval()
    learner = learner_cls(model, cfg)

    if dm is None:
        dm = SSLMAEDataModule(cfg)
        dm.setup()

    trainer = L.Trainer(accelerator='auto', devices=[0], precision=cfg.training.precision,
                        enable_model_summary=False, logger=False)
    logits_list = trainer.predict(learner, datamodule=dm)
    logits = torch.cat(logits_list, dim=0).cpu().float()
    all_probas[mode] = torch.sigmoid(logits).numpy() if cfg.data.learning_task == 'mlc' \
                       else torch.softmax(logits, dim=1).numpy()
    print(f"  {MODE_LABELS[mode]:25s} predictions ready")

# Collect targets and images
targets = collect_targets(dm.predict_dataloader())
test_batches = list(dm.predict_dataloader())
all_images = torch.cat([b[0] for b in test_batches], dim=0)
all_raw = torch.cat([b[1] for b in test_batches], dim=0)

print(f"\nAll models loaded. {len(all_images)} test images ready.")

## 8. Compare predictions (re-run for random samples)

In [ ]:
N_SHOW = 4
n_modes = len(all_probas)
active_modes = list(all_probas.keys())

# Random sample indices (re-run cell for different images)
indices = np.random.choice(len(all_images), N_SHOW, replace=False)

fig, axes = plt.subplots(1 + n_modes, N_SHOW, figsize=(3.5 * N_SHOW, 3 * (1 + n_modes)))

for col, idx in enumerate(indices):
    # Row 0: image + true labels
    img = unnormalize(all_images[idx])
    axes[0, col].imshow(img)
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    # True labels
    t = targets[idx].numpy()
    if t.ndim > 0:  # multi-label
        true_classes = [CLASS_NAMES[i] for i in range(len(t)) if t[i] > 0]
    else:
        true_classes = [CLASS_NAMES[int(t)]]
    true_str = ", ".join(true_classes) if true_classes else "none"
    axes[0, col].set_title(true_str, fontsize=8, color="#333", wrap=True)
    if col == 0:
        axes[0, col].set_ylabel("Image\n+ true labels", fontsize=9)

    # Rows 1+: predictions per mode
    for row, mode in enumerate(active_modes):
        ax = axes[1 + row, col]
        p = all_probas[mode][idx]
        top_k = np.argsort(p)[-5:][::-1]
        top_names = [CLASS_NAMES[k] for k in top_k]
        top_vals = [p[k] for k in top_k]

        # Color: green if in true labels, gray otherwise
        bar_colors = ['#2E86AB' if t[k] > 0 else '#ccc' for k in top_k]

        ax.barh(range(len(top_k)), top_vals, color=bar_colors)
        ax.set_yticks(range(len(top_k)))
        ax.set_yticklabels(top_names, fontsize=8)
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        if col == 0:
            ax.set_ylabel(MODE_LABELS[mode], fontsize=9)

plt.suptitle(f"{MODEL_TYPE.upper()} — predictions ($f_l$={int(PRED_FRAC*100)}%)\n"
             "Blue = correct class, gray = incorrect", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()